In [1]:
import pickle
from pathlib import Path
import pandas as pd
from scipy import stats

Opens WESAD folder and opens every .pkl file to collect wrist data for each patient before concatenating into a singular dataset. Final dataframe has a column for each participant's signal and emotion label at recording. Labels were downsampled from 700hz to 4hz by taking the mode of each divided section of the signal.

In [ ]:
# Set directory with WESAD data here
wesad_dir = Path("../Data/wesad_eda/WESAD")
# Empty list to store all the participants
subject_list = []
# Loop to open all the files
for i in wesad_dir.rglob("*.pkl"):
    subject_id = i.parent.name
    with open(i, "rb") as file:
        data = pickle.load(file, encoding='latin1')
        wrist_data = data['signal']['wrist']
        eda_signal = wrist_data['EDA'].flatten()
        #subject_id = data['label']
        # APPARENTLY the label is for 700hz and this needs to be recounted to match labels to 4hz accordingly
        label_700hz = data['label'].flatten()
        # 700/4=175 so we take chunks every 175 then label based on mode rule, I am open to suggestions for downsampling the labels
        n_chunks = len(label_700hz) // 175
        label_chunks = label_700hz[: n_chunks * 175].reshape(n_chunks, 175)
        label_4hz = stats.mode(label_chunks, axis=1).mode.flatten()
        # Rounding issue with getting mismatched vector lengths for concatenation
        min_len = min(len(eda_signal), len(label_4hz))
        # Build Dataframe
        df_subject = pd.DataFrame({
            'EDA': eda_signal[:min_len],
            'Label': label_4hz[:min_len]
            })
        df_subject.columns = [subject_id + '_EDA', subject_id + '_Label']
        #df_subject['Subject'] = subject_id
        subject_list.append(df_subject)
#Make the final dataframe        
eda_final = pd.concat(subject_list, axis=1)
eda_final = eda_final.dropna()

,S15_EDA,S15_Label,S7_EDA,S7_Label,S3_EDA,S3_Label,S2_EDA,S2_Label,S14_EDA,S14_Label,...,S4_EDA,S4_Label,S13_EDA,S13_Label,S5_EDA,S5_Label,S9_EDA,S9_Label,S6_EDA,S6_Label
0,0.220298,0.0,5.969716,0.0,2.248749,0.0,1.138257,0.0,0.277864,0.0,...,0.169117,0.0,9.149161,0.0,0.547723,0.0,0.329014,0.0,2.779563,0
1,0.170407,0.0,6.100200,0.0,2.710001,0.0,1.125444,0.0,0.266351,0.0,...,0.164001,0.0,8.816556,0.0,0.481218,0.0,0.326456,0.0,2.785958,0
2,0.189596,0.0,6.096362,0.0,2.554969,0.0,1.011405,0.0,0.195992,0.0,...,0.167838,0.0,8.713039,0.0,0.518239,0.0,0.336689,0.0,2.761658,0
3,0.151218,0.0,6.088686,0.0,2.818907,0.0,1.033188,0.0,0.226694,0.0,...,0.155047,0.0,8.687454,0.0,0.440223,0.0,0.332852,0.0,2.729684,0
4,0.166569,0.0,6.114272,0.0,2.630563,0.0,0.935807,0.0,0.126912,0.0,...,0.158884,0.0,8.746300,0.0,0.391623,0.0,0.327735,0.0,2.693874,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20887,0.660360,0.0,4.696972,0.0,0.397456,3.0,0.154027,0.0,0.314962,4.0,...,0.240751,0.0,5.592779,4.0,1.137415,0.0,0.499144,0.0,1.437967,0
20888,0.671873,0.0,4.696972,0.0,0.397456,3.0,0.148901,0.0,0.316242,4.0,...,0.240751,0.0,5.594059,4.0,1.139973,0.0,0.494028,0.0,1.435409,0
20889,0.665477,0.0,4.698251,0.0,0.398737,3.0,0.175809,0.0,0.316242,4.0,...,0.239472,0.0,5.592779,4.0,1.136136,0.0,0.496586,0.0,1.435409,0
20890,0.651405,0.0,4.690576,0.0,0.397456,3.0,0.177091,0.0,0.316242,4.0,...,0.239472,0.0,5.595337,4.0,1.138694,0.0,0.497865,0.0,1.435409,0


Save final dataset to a csv file.

In [8]:
eda_final.to_csv("eda_final.csv", index=False)

```
# Relabel numbers for easy human readable labels
final_df.loc[final_df['Label'] == 0, 'Label'] = 'Transient'
final_df.loc[final_df['Label'] == 1, 'Label'] = 'Baseline'
final_df.loc[final_df['Label'] == 2, 'Label'] = 'Stress'
final_df.loc[final_df['Label'] == 3, 'Label'] = 'Amusement'
final_df.loc[final_df['Label'] == 4, 'Label'] = 'Meditation'
final_df.loc[final_df['Label'] == 5, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 6, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 7, 'Label'] = 'Ignore'
final_df['Label'].unique()
```

We see each subject's EDA has different length.

In [6]:
for i in range(len(subject_list)):
    print(f"{len(subject_list[i])}")

21008
20952
25972
24316
22192
21864
20932
23680
21984
22524
25692
22148
25032
20892
28284
